In [1]:
import time
start = time.perf_counter()

import pandas as pd
import unicodedata
import re
import numpy as np

# --- Configurações Iniciais e Funções Auxiliares ---

def remove_accents(s):
    if not isinstance(s, str):
        return s
    s = unicodedata.normalize('NFD', s)
    s = "".join(c for c in s if unicodedata.category(c) != "Mn")
    return s.replace("ç", "c").replace("Ç", "C")

# --- Carregamento de Dados ---
# Pandas lê arquivos parquet nativamente se tiver 'pyarrow' ou 'fastparquet' instalado
df_diesel_cng = pd.read_parquet("../data/fuel_prices/bronze/Diesel and CNG Prices.parquet")
df_lpg = pd.read_parquet("../data/fuel_prices/bronze/LPG Prices.parquet")
df_gasoline_ethanol = pd.read_parquet("../data/fuel_prices/bronze/Gasoline and Ethanol Prices.parquet")

# Union (Concatenar DataFrames)
df = pd.concat([df_diesel_cng, df_lpg, df_gasoline_ethanol], ignore_index=True)

# --- Transformações ---

# 1. Drop de colunas
cols_to_drop = ['CNPJ da Revenda', 'Nome da Rua', 'Numero Rua', 'Complemento', 'Cep', 'Valor de Compra']
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

# 2. Renomear colunas
new_columns = {
    df.columns[0]: 'nm_region', df.columns[1]: 'nm_state', df.columns[2]: 'nm_city',
    df.columns[3]: 'nm_gas_station', df.columns[4]: 'nm_neighborhood', df.columns[5]: 'nm_fuel_type',
    df.columns[6]: 'dt_date', df.columns[7]: 'nu_fuel_price', df.columns[8]: 'nm_unit_of_measurement',
    df.columns[9]: 'nm_fuel_brand'
}
df = df.rename(columns=new_columns)
df

,nm_region,nm_state,nm_city,nm_gas_station,nm_neighborhood,nm_fuel_type,dt_date,nu_fuel_price,nm_unit_of_measurement,nm_fuel_brand
0,N,AC,RIO BRANCO,AUTO POSTO AMAPA - EIRELI,AREAL,DIESEL,03/01/2022,"6,09",R$ / litro,VIBRA ENERGIA
1,N,AC,RIO BRANCO,AUTO POSTO AMAPA - EIRELI,AREAL,DIESEL S10,03/01/2022,"6,12",R$ / litro,VIBRA ENERGIA
2,N,AC,RIO BRANCO,AUTO POSTO ACAUAN LTDA,VILA ACRE,DIESEL,03/01/2022,"6,09",R$ / litro,VIBRA ENERGIA
3,N,AC,RIO BRANCO,AUTO POSTO ACAUAN LTDA,VILA ACRE,DIESEL S10,03/01/2022,"6,12",R$ / litro,VIBRA ENERGIA
4,N,AC,RIO BRANCO,AUTO POSTO CORRENTAO LTDA,SANTA INES,DIESEL,03/01/2022,"6,08",R$ / litro,BRANCA
...,...,...,...,...,...,...,...,...,...,...
3814394,SE,MG,BARBACENA,POSTO DAS FLORES LTDA,PONTILHAO,GASOLINA ADITIVADA,28/02/2026,"6,59",R$ / litro,RAIZEN
3814395,SE,MG,BARBACENA,POSTO DAS FLORES LTDA,PONTILHAO,ETANOL,28/02/2026,"4,59",R$ / litro,RAIZEN
3814396,SE,MG,BARBACENA,POSTO SETE DE SETEMBRO LTDA,CENTRO,GASOLINA,28/02/2026,"6,19",R$ / litro,RAIZEN
3814397,SE,MG,BARBACENA,POSTO SETE DE SETEMBRO LTDA,CENTRO,GASOLINA ADITIVADA,28/02/2026,"6,49",R$ / litro,RAIZEN


In [2]:
df['dt_date'] = pd.to_datetime(df['dt_date'], format='%d/%m/%Y')
df

,nm_region,nm_state,nm_city,nm_gas_station,nm_neighborhood,nm_fuel_type,dt_date,nu_fuel_price,nm_unit_of_measurement,nm_fuel_brand
0,N,AC,RIO BRANCO,AUTO POSTO AMAPA - EIRELI,AREAL,DIESEL,2022-01-03,"6,09",R$ / litro,VIBRA ENERGIA
1,N,AC,RIO BRANCO,AUTO POSTO AMAPA - EIRELI,AREAL,DIESEL S10,2022-01-03,"6,12",R$ / litro,VIBRA ENERGIA
2,N,AC,RIO BRANCO,AUTO POSTO ACAUAN LTDA,VILA ACRE,DIESEL,2022-01-03,"6,09",R$ / litro,VIBRA ENERGIA
3,N,AC,RIO BRANCO,AUTO POSTO ACAUAN LTDA,VILA ACRE,DIESEL S10,2022-01-03,"6,12",R$ / litro,VIBRA ENERGIA
4,N,AC,RIO BRANCO,AUTO POSTO CORRENTAO LTDA,SANTA INES,DIESEL,2022-01-03,"6,08",R$ / litro,BRANCA
...,...,...,...,...,...,...,...,...,...,...
3814394,SE,MG,BARBACENA,POSTO DAS FLORES LTDA,PONTILHAO,GASOLINA ADITIVADA,2026-02-28,"6,59",R$ / litro,RAIZEN
3814395,SE,MG,BARBACENA,POSTO DAS FLORES LTDA,PONTILHAO,ETANOL,2026-02-28,"4,59",R$ / litro,RAIZEN
3814396,SE,MG,BARBACENA,POSTO SETE DE SETEMBRO LTDA,CENTRO,GASOLINA,2026-02-28,"6,19",R$ / litro,RAIZEN
3814397,SE,MG,BARBACENA,POSTO SETE DE SETEMBRO LTDA,CENTRO,GASOLINA ADITIVADA,2026-02-28,"6,49",R$ / litro,RAIZEN


In [ ]:
df['dt_year'] = df['dt_date'].dt.year.astype(str).replace('\.0', '', regex=True)
df['dt_month'] = df['dt_date'].dt.month.astype(str).replace('\.0', '', regex=True)
df

df['dt_year'] = np.where(df['dt_year'] == 'nan', np.nan, df['dt_year'])
df['dt_month'] = np.where(df['dt_month'] == 'nan', np.nan, df['dt_month'])

In [4]:
df

,nm_region,nm_state,nm_city,nm_gas_station,nm_neighborhood,nm_fuel_type,dt_date,nu_fuel_price,nm_unit_of_measurement,nm_fuel_brand,dt_year,dt_month
0,N,AC,RIO BRANCO,AUTO POSTO AMAPA - EIRELI,AREAL,DIESEL,2022-01-03,"6,09",R$ / litro,VIBRA ENERGIA,2022,1
1,N,AC,RIO BRANCO,AUTO POSTO AMAPA - EIRELI,AREAL,DIESEL S10,2022-01-03,"6,12",R$ / litro,VIBRA ENERGIA,2022,1
2,N,AC,RIO BRANCO,AUTO POSTO ACAUAN LTDA,VILA ACRE,DIESEL,2022-01-03,"6,09",R$ / litro,VIBRA ENERGIA,2022,1
3,N,AC,RIO BRANCO,AUTO POSTO ACAUAN LTDA,VILA ACRE,DIESEL S10,2022-01-03,"6,12",R$ / litro,VIBRA ENERGIA,2022,1
4,N,AC,RIO BRANCO,AUTO POSTO CORRENTAO LTDA,SANTA INES,DIESEL,2022-01-03,"6,08",R$ / litro,BRANCA,2022,1
...,...,...,...,...,...,...,...,...,...,...,...,...
3814394,SE,MG,BARBACENA,POSTO DAS FLORES LTDA,PONTILHAO,GASOLINA ADITIVADA,2026-02-28,"6,59",R$ / litro,RAIZEN,2026,2
3814395,SE,MG,BARBACENA,POSTO DAS FLORES LTDA,PONTILHAO,ETANOL,2026-02-28,"4,59",R$ / litro,RAIZEN,2026,2
3814396,SE,MG,BARBACENA,POSTO SETE DE SETEMBRO LTDA,CENTRO,GASOLINA,2026-02-28,"6,19",R$ / litro,RAIZEN,2026,2
3814397,SE,MG,BARBACENA,POSTO SETE DE SETEMBRO LTDA,CENTRO,GASOLINA ADITIVADA,2026-02-28,"6,49",R$ / litro,RAIZEN,2026,2


In [4]:
import pandas as pd
import requests
import time

# 1. Configurações e Leitura
# No Pandas, lemos o parquet diretamente (precisa das libs 'pyarrow' ou 'fastparquet' instaladas)
df = pd.read_parquet("../data/fuel_prices/silver/fuels_prices")
df

,nm_region,nm_state,nm_city,nm_gas_station,nm_neighborhood,nm_fuel_type,dt_date,nu_fuel_price,nm_unit_of_measurement,nm_fuel_brand,dt_year,dt_month,ab_state,uf_city
0,Norte,Acre,Rio Branco,Auto Posto Amapa - Eireli,Areal,Diesel,2022-01-03,6.09,R$ / litro,Vibra Energia,2022,1,AC,ac_rio_branco
1,Norte,Acre,Rio Branco,Auto Posto Amapa - Eireli,Areal,Diesel S10,2022-01-03,6.12,R$ / litro,Vibra Energia,2022,1,AC,ac_rio_branco
2,Norte,Acre,Rio Branco,Auto Posto Acauan Ltda,Vila Acre,Diesel,2022-01-03,6.09,R$ / litro,Vibra Energia,2022,1,AC,ac_rio_branco
3,Norte,Acre,Rio Branco,Auto Posto Acauan Ltda,Vila Acre,Diesel S10,2022-01-03,6.12,R$ / litro,Vibra Energia,2022,1,AC,ac_rio_branco
4,Norte,Acre,Rio Branco,Auto Posto Correntao Ltda,Santa Ines,Diesel,2022-01-03,6.08,R$ / litro,Branca,2022,1,AC,ac_rio_branco
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3197862,Sudeste,Minas Gerais,Barbacena,Posto Das Flores Ltda,Pontilhao,Gasolina Aditivada,2026-02-28,6.59,R$ / litro,Raizen,2026,2,MG,mg_barbacena
3197863,Sudeste,Minas Gerais,Barbacena,Posto Das Flores Ltda,Pontilhao,Etanol,2026-02-28,4.59,R$ / litro,Raizen,2026,2,MG,mg_barbacena
3197864,Sudeste,Minas Gerais,Barbacena,Posto Sete De Setembro Ltda,Centro,Gasolina,2026-02-28,6.19,R$ / litro,Raizen,2026,2,MG,mg_barbacena
3197865,Sudeste,Minas Gerais,Barbacena,Posto Sete De Setembro Ltda,Centro,Gasolina Aditivada,2026-02-28,6.49,R$ / litro,Raizen,2026,2,MG,mg_barbacena
